# Agent Evaluation

Evaluate the Credit Risk Scorecard AI Agent using LLM-as-judge against 7 quality criteria.

Run `data_gen.ipynb` first to generate interaction logs before running this notebook.

In [1]:
import sys
sys.path.insert(0, '../app')

import json
import pandas as pd
from pathlib import Path
from pydantic import BaseModel
from pydantic_ai import Agent
from tqdm.auto import tqdm
from dotenv import load_dotenv

load_dotenv('../.env', override=True)

LOG_DIR = Path('../app/logs')

In [2]:
evaluation_prompt = """
Use this checklist to evaluate the quality of an AI agent's answer (<ANSWER>) to a user question (<QUESTION>).
We also include the entire log (<LOG>) for analysis.

For each item, check if the condition is met.

Checklist:
- instructions_follow: The agent followed the system prompt instructions
- instructions_avoid: The agent avoided doing things it was told not to do
- answer_relevant: The response directly addresses the user's question
- answer_clear: The answer is clear, accurate, and correct
- answer_citations: The answer includes references or citations to source material
- completeness: The answer covers all key aspects of the question
- tool_call_search: The search tool was actually invoked to find the answer
""".strip()

user_prompt_format = """
<INSTRUCTIONS>
{instructions}
</INSTRUCTIONS>

<QUESTION>
{question}
</QUESTION>

<ANSWER>
{answer}
</ANSWER>

<LOG>
{log}
</LOG>
""".strip()

In [3]:
class EvaluationCheck(BaseModel):
    check_name: str
    justification: str
    check_pass: bool

class EvaluationChecklist(BaseModel):
    checklist: list[EvaluationCheck]
    summary: str

eval_agent = Agent(
    name='eval_agent',
    model='gpt-4o-mini',
    instructions=evaluation_prompt,
    output_type=EvaluationChecklist
)

In [4]:
def load_log_file(log_file):
    with open(log_file, 'r') as f:
        data = json.load(f)
        data['log_file'] = log_file
        return data

def simplify_log_messages(messages):
    log_simplified = []
    for m in messages:
        parts = []
        for original_part in m['parts']:
            part = original_part.copy()
            kind = part['part_kind']
            if kind == 'user-prompt':
                part.pop('timestamp', None)
            if kind in ('tool-call', 'tool-return'):
                part.pop('tool_call_id', None)
            if kind == 'tool-return':
                content = part.get('content', '')
                if isinstance(content, list):
                    part['content'] = [{'filename': c.get('filename', ''), 'content': c.get('content', '')[:200]} for c in content]
            parts.append(part)
        entry = {'kind': m['kind'], 'parts': parts}
        log_simplified.append(entry)
    return log_simplified

async def evaluate_log_record(eval_agent, log_record):
    messages = log_record['messages']
    instructions = log_record['system_prompt']
    question = messages[0]['parts'][0]['content']
    answer = messages[-1]['parts'][0]['content']
    log_simplified = simplify_log_messages(messages)
    log = json.dumps(log_simplified)
    user_prompt = user_prompt_format.format(
        instructions=instructions,
        question=question,
        answer=answer,
        log=log
    )
    result = await eval_agent.run(user_prompt, output_type=EvaluationChecklist)
    return result.output

In [5]:
# Load all credit risk agent logs
log_files = sorted(LOG_DIR.glob('credit_risk_agent_*.json'))
print(f'Found {len(log_files)} log files')

eval_set = [load_log_file(f) for f in log_files]
print(f'Loaded {len(eval_set)} interactions for evaluation')

Found 16 log files
Loaded 16 interactions for evaluation


In [6]:
# Run evaluation on all interactions
eval_results = []

for log_record in tqdm(eval_set):
    eval_result = await evaluate_log_record(eval_agent, log_record)
    eval_results.append((log_record, eval_result))

print(f'Evaluated {len(eval_results)} interactions')

  0%|          | 0/16 [00:00<?, ?it/s]

Evaluated 16 interactions


In [7]:
# Build results dataframe
rows = []
for log_record, eval_result in eval_results:
    question = log_record['messages'][0]['parts'][0]['content']
    answer = log_record['messages'][-1]['parts'][0]['content']
    row = {
        'log_file': log_record['log_file'].name,
        'question': question[:60] + '...',
        'answer': answer[:60] + '...',
    }
    for check in eval_result.checklist:
        row[check.check_name] = check.check_pass
    rows.append(row)

df_evals = pd.DataFrame(rows)
df_evals.head()

,log_file,question,answer,instructions_follow,instructions_avoid,answer_relevant,answer_clear,answer_citations,completeness,tool_call_search
0,credit_risk_agent_20260213_063120_42175b.json,How do I create WoE bins for a credit feature?...,To create Weight of Evidence (WoE) bins for a ...,True,True,True,True,True,True,True
1,credit_risk_agent_20260213_063232_d3b786.json,How do i qualitfy for a credit?...,Qualifying for credit generally involves sever...,True,True,True,True,False,True,False
2,credit_risk_agent_20260213_063259_6c0d36.json,what model are you running on?...,"I am an AI language model developed by OpenAI,...",False,True,False,False,False,False,False
3,credit_risk_agent_20260213_063340_f2626a.json,"explain these term better for me\n\n WoE, IV, ...","Here's a breakdown of the terms **WoE**, **IV*...",True,True,True,True,True,True,True
4,credit_risk_agent_20260213_063516_d6fc93.json,What is Population Stability Index (PSI) and h...,The **Population Stability Index (PSI)** is a ...,True,True,True,True,True,True,True


In [8]:
# Overall pass rates
print('=== Overall Pass Rates ===')
df_evals.mean(numeric_only=True)

=== Overall Pass Rates ===


instructions_follow    0.8750
instructions_avoid     1.0000
answer_relevant        0.8750
answer_clear           0.8750
answer_citations       0.8125
completeness           0.8750
tool_call_search       0.8125
dtype: float64